# 00. Среда, угрозы и протокол

Вход: данные PersonPath22 и новая папка кода. Выход: отдельный run_id, конфигурация, правила независимой оценки.

Подключение после каждого сброса Colab. Загрузите всю папку `notebooks_colab_full`
в `MyDrive/ML6/`. Исходные видео и старые результаты остаются в прежних папках.
Вычисления читают локальные кадры; результаты нового запуска пишутся в `runs/RUN_ID/`.
В 00 удаляются только 20 указанных резервных видео и карантинный ролик. В 01 исходные
train-MP4 удаляются после проверки архивов всех аннотированных чистых кадров.

In [ ]:
import sys, os, subprocess, importlib.util
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules or Path('/content').is_dir()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/ML6')
else:
    PROJECT = Path('D:/ML6')
RUN_ID = 'blue_aug_20261112_v2'  # одинаковый во всех 10 ноутбуках; старый run не меняем
BUNDLE = PROJECT / 'notebooks_colab_full'
assert (BUNDLE / 'tracking_lab.py').is_file(), BUNDLE
packages = {'scipy':'scipy', 'pandas':'pandas', 'cv2':'opencv-python-headless',
    'matplotlib':'matplotlib', 'tqdm':'tqdm', 'trackeval':'trackeval==1.3.0',
    'cryptography':'cryptography', 'imageio':'imageio', 'imageio_ffmpeg':'imageio-ffmpeg'}
missing = [p for m,p in packages.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
# Совместимую пару torch/torchvision предоставляет Colab; автоматически её не заменяем.
sys.path.insert(0, str(BUNDLE))
import tracking_lab as tl
assert Path(tl.__file__).resolve() == (BUNDLE / 'tracking_lab.py').resolve(), 'Перезапустите runtime: загружен старый модуль'
assert tl.SCHEMA == 4, 'Перезапустите runtime после обновления исходного кода'
import torch, pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display, Video, HTML, Image
lab = tl.Lab(PROJECT, RUN_ID)
torch.hub.set_dir(str(PROJECT / 'pretrained_cache'))
display(pd.Series(lab.environment(), name='Среда'))
print('Запуск:', RUN_ID, '| Результаты:', lab.root)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Среда
python,"3.13.16 (main, Oct 1 2026, 15:40:24) [GCC 13...."
packages,"{'torch': '2.14.1+cu130', 'torchvision': '0.29..."
device,cuda
gpu,NVIDIA A100-SXM4-40GB


Запуск: blue_aug_20261112_v2 | Результаты: /content/drive/MyDrive/ML6/runs/blue_aug_20261112_v2


Аудит прежнего прогона `robust_11gib_v1`: 20 train, 4 val, 5 test, 1714 train-кадров,
3 эпохи детектора и 5×50 шагов ReID. 08 завершился экспортом ZIP. Это законченный
малый эксперимент, а не полноразмерное обучение. HOTA clean=0,331590 → PGD=0,079048;
IDF1 0,372060 → 0,049472. Внешне слабые patch/ReID-атаки требуют проверки силы.

Новый профиль использует все пригодные train-видео кроме 12 validation; все аннотированные
train-кадры сохраняются в чистые JPEG-архивы до удаления MP4. Для оценки берутся все
аннотированные позиции. 5 старых test-видео — development; 20 других — новый final.
По решению пользователя считаем новый набор независимым; это допущение записано
в exposure ledger. Дубликаты объединяются в группы до разбиения.

In [ ]:
# Меняйте профиль только ДО 01. При другом бюджете после 01 задайте новый RUN_ID.
display(pd.Series(lab.cfg, name='Параметр'))
lab.save_config()
assert lab.cfg['run_name'] != 'robust_11gib_v1'

,Параметр
schema,4
run_name,blue_aug_20261112_v2
seed,42
val_videos,12
final_videos,20
train_frames_per_video,None
reserve_remove_count,20
retire_train_mp4,True
augment,"{'mixture': {'clean': 0.4, 'photo': 0.15, 'jpe..."
eval_frames_per_video,None


Удаление источников по вашему указанию: сначала 20 из 40 резервных видео и 1 из
карантина (MP4 и соответствующие per-video JSON). Имена закреплены в исходном
разбиении и не выбираются повторно после удаления. Train/val/dev/final не меняются.
Source splits.json, веса и результаты сохраняются. Ниже виден полный список,
после него выполняется удаление с SHA256-проверкой и журналом возобновления.

In [ ]:
SOURCE=PROJECT/'tracking-dataset-main/dataset/personpath22'
DATA_SOURCE_RUN='blue_aug_20261112_v2'  # общий clean-архив при новом RUN_ID и новом рецепте
if RUN_ID!=DATA_SOURCE_RUN and not tl.lifecycle.prepared_data_available(lab):
    source_manifest=PROJECT/'runs'/DATA_SOURCE_RUN/'data/manifests/frames_meta.json'
    if source_manifest.exists():
        display(pd.Series(tl.reuse_prepared_data(lab,DATA_SOURCE_RUN)))
reference=tl.load_json(BUNDLE/'audit/reference_splits_before_pruning.json')
source_audit=tl.load_json(BUNDLE/'audit/local_dataset.json')
pruning=tl.lifecycle.build_pruning_plan(reference,source_audit,lab.cfg['reserve_remove_count'])
display(pd.DataFrame(pruning['entries']).rename(columns={'relative':'Удаляемый файл','video':'Видео','sha256':'Ожидаемый SHA256'}))
display(pd.DataFrame([{'Набор':s,'Останется видео':len(v)} for s,v in pruning['remaining_splits'].items()]))

,Удаляемый файл,Ожидаемый SHA256,Видео
0,raw_data/uid_vid_00024.mp4,c83019fbe473885c9b833b69f41018c10964cacaa819fc...,uid_vid_00024.mp4
1,annotation/anno_visible_2022/uid_vid_00024.mp4...,e1bc460507203b1451ea313216d70315c331c3881628b9...,uid_vid_00024.mp4
2,raw_data/uid_vid_00069.mp4,bedeb25e5348237f2f4451a83528c07f7480edf7573330...,uid_vid_00069.mp4
3,annotation/anno_visible_2022/uid_vid_00069.mp4...,a28bb424acb1b816db2e4acb1ab9f7efa5669997eae798...,uid_vid_00069.mp4
4,raw_data/uid_vid_00212.mp4,c3038c5b812b48c368a47eb6bbba12cbf43f1151744ef0...,uid_vid_00212.mp4
5,annotation/anno_visible_2022/uid_vid_00212.mp4...,98f42c4a146cdb4dfb41ceebcf9444284ac335a6cf6e51...,uid_vid_00212.mp4
6,raw_data/uid_vid_00078.mp4,4c3be5895acc8cf634bf7a27c64a38d8cd415c3ef1f958...,uid_vid_00078.mp4
7,annotation/anno_visible_2022/uid_vid_00078.mp4...,080a248cd93e0f0355b6823e9669f807888241e0619d32...,uid_vid_00078.mp4
8,raw_data/uid_vid_00158.mp4,54e81996b7c9ad321f55dc54994b431a460d956c8dd9a6...,uid_vid_00158.mp4
9,annotation/anno_visible_2022/uid_vid_00158.mp4...,7726aa704dfa89c9c1633ce6884ebb2e34b0c1b97a7841...,uid_vid_00158.mp4


,Набор,Останется видео
0,train,94
1,val,12
2,dev,5
3,final,20
4,quarantine,0
5,reserve,20


In [ ]:
# Выполняется ровно запрошенная очистка. Все хеши проверяются ДО первого удаления.
journal=tl.lifecycle.prune_reserve_and_quarantine(lab,SOURCE,pruning)
print('Очистка завершена:',journal['complete'],'| файлов:',len(journal['entries']))

SHA256 до удаления:   0%|          | 0/42 [00:00<?, ?файл/s]

Удаление по проверенному списку:   0%|          | 0/42 [00:00<?, ?файл/s]

Очистка завершена: True | файлов: 42


White-box: атакующий знает код, веса, конфигурацию и имеет градиенты. Evasion меняет
только RGB-пиксели [0,1]. PGD: L∞ ≤ 4/255, 20 шагов × 3 рестарта. FGSM: 1 шаг.
Патч: ≤12% площади одной видимой рамки и ≤12% кадра, значения внутри [0,1], EOT
яркость/blur. Это цифровой локальный патч, не доказанная атака на печатную одежду.
ReID: pull/push ближайшей пары людей, L∞ ≤4/255; кадры без пары не считаются успехом.
Во всех атакованных последовательностях меняется центральное окно 30–60% кадров:
число атакованных кадров = ceil(0,6N)−floor(0,3N). До/после него измеряется история ID.

Цели: пропуск человека, ложные обнаружения, ID switches и сохранение неправильного
ID после воздействия. Poisoning требует записи train-разметки и рассматривается
отдельно. Подмена весов требует записи артефактов; защищаем её хешами и подписью.
Разработчик выбирает параметры по validation, затем фиксирует релиз. Независимая
Red Team получает релиз и final; новые улучшения после её результатов — новый run_id.

In [ ]:
threats = pd.DataFrame([
    ['PGD/FGSM','evasion','веса, код, градиенты','L∞=4/255; окно 30–60%','обнаружение'],
    ['patch/EOT','evasion','пиксели одной области','≤12% рамки и кадра','скрытие человека'],
    ['ReID pull/push','evasion','градиенты ReID','L∞=4/255; нужна пара','подмена ID'],
    ['poisoning','train','запись данных','отдельная копия/аудит','ошибочное обучение'],
    ['подмена релиза','artifact','запись файлов','доверенный публичный ключ','подмена модели'],
],columns=['Сценарий','Этап','Доступ','Ограничение','Цель'])
display(threats)
tl.save_json(lab.path('configs/threat_model.json'), threats.to_dict('records'))

,Сценарий,Этап,Доступ,Ограничение,Цель
0,PGD/FGSM,evasion,"веса, код, градиенты",L∞=4/255; окно 30–60%,обнаружение
1,patch/EOT,evasion,пиксели одной области,≤12% рамки и кадра,скрытие человека
2,ReID pull/push,evasion,градиенты ReID,L∞=4/255; нужна пара,подмена ID
3,poisoning,train,запись данных,отдельная копия/аудит,ошибочное обучение
4,подмена релиза,artifact,запись файлов,доверенный публичный ключ,подмена модели


PosixPath('/content/drive/MyDrive/ML6/runs/blue_aug_20261112_v2/configs/threat_model.json')

Сравниваем `plain`, `natural`, `adversarial`: одинаковые исходные веса, видео,
порядок frame slots, P×K-сэмплирование и число optimizer steps. Для каждого примера
выбирается одно воздействие. Финальная смесь: 40% clean, 15% photo, 10% JPEG,
5% noise, 5% blur, 5% geometry, 15% PGD, 5% occlusion/patch. Это стартовый рецепт,
его пользу проверяет validation. PGD включается после одной эпохи адаптации и
нарастает за две эпохи; пока он выключен, его доля переносится в clean.
Adversarial добавляет PGD, поэтому его wall-clock больше; сравниваем
и фактические GPU-часы. «Эпоха» детектора — 1500 сбалансированных frame slots,
а не обход каждого JPEG. Полный сравнительный запуск может занять несколько сессий.
В 02 измеряется стоимость одного шага. Уменьшение бюджета требует нового run_id.

In [ ]:
display(tl.cd.mixture_table(lab.cfg))
tl.save_json(lab.path('configs/augmentation.json'),lab.cfg['augment'])

,Эпоха (дробная),Воздействие,"Доля, %"
0,0.0,Обычные,55.00
1,0.0,Яркость / контраст / гамма,15.00
2,0.0,JPEG,10.00
3,0.0,Шум,5.00
4,0.0,Размытие,5.00
5,0.0,Геометрия,5.00
6,0.0,PGD,0.00
7,0.0,Перекрытие,5.00
8,1.5,Обычные,51.25
9,1.5,Яркость / контраст / гамма,15.00


PosixPath('/content/drive/MyDrive/ML6/runs/blue_aug_20261112_v2/configs/augmentation.json')

Продолжение после отключения: повторите bootstrap. Журнал очистки сохранён; завершённое удаление не выбирает новые файлы. Далее 01.